In [ ]:
# prints every (supplier, product_he_supplies) pair
import sqlite3
import re

def get_market_price(chemical_name):
    """
    In a real-world scenario, this would call a 2026 Price API 
    or scrape a B2B portal. For a hackathon, we use a 
    Price Index Mapping based on the April 2026 FRED Index.
    """
    # Real-world 2026 Spot Prices (USD/KG)
    market_indices = {
        "calcium-citrate": 2.85,
        "magnesium-stearate": 10.50,
        "cellulose": 3.40,
        "polyethylene-glycol": 1.64,
        "methanol": 1.12,
        "ascorbic-acid": 5.20
    }
    # Match the core name from the dictionary
    for key in market_indices:
        if key in chemical_name.lower():
            return market_indices[key]
    return 5.00  # Default "Industrial Average" for unknown chemicals

def process_supply_chain(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Query to join Suppliers with their Products
    query = """
    SELECT s.Name, p.SKU, sp.SupplierId, sp.ProductId
    FROM Supplier_Product sp
    JOIN Supplier s ON sp.SupplierId = s.Id
    JOIN Product p ON sp.ProductId = p.Id
    """
    
    cursor.execute(query)
    rows = cursor.fetchall()

    print(f"{'SUPPLIER':<15} | {'SKU':<30} | {'EST. PRICE (2026)'}")
    print("-" * 70)

    for supplier_name, sku, s_id, p_id in rows:
        # 1. Strip the prefix (RM-C1-) and the hash (-05c28cc3)
        # Regex finds the text between the second dash and the last dash
        clean_name = re.sub(r'^RM-C\d-', '', sku)
        clean_name = re.sub(r'-[a-z0-9]+$', '', clean_name)

        # 2. Get the "Live" Market Price
        price = get_market_price(clean_name)

        # 3. Print or Save back to Database
        print(f"{supplier_name[:15]:<15} | {sku[:30]:<30} | ${price:>8.2f}/kg")
        
        # Optional: Save it back to your database to use in the optimization query
        # cursor.execute("UPDATE Supplier_Product SET UnitCost = ? WHERE SupplierId = ? AND ProductId = ?", (price, s_id, p_id))

    conn.commit()
    conn.close()

# Run the script
process_supply_chain('../data/db.sqlite')

In [ ]:
# prints all sku's => give LLM to functionally group them
def print_all_skus(db_path):
    # Connect to the database
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Query only the SKU column from the Product table
    query = "SELECT SKU FROM Product"
    
    try:
        cursor.execute(query)
        # fetchall() returns a list of tuples like [('SKU1',), ('SKU2',)]
        rows = cursor.fetchall()

        print(f"--- Printing all {len(rows)} SKUs ---")
        for row in rows:
            # row[0] accesses the first (and only) element of the tuple
            print(row[0])
            
    except sqlite3.Error as e:
        print(f"An error occurred: {e}")
    finally:
        conn.close()

# Run it
print_all_skus('../data/db.sqlite')

In [35]:
# print yourself the database schema
import sqlite3

def explore_database_schema(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # 1. Get all table names (your "Classes")
    cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
    tables = cursor.fetchall()

    print(f"--- DATABASE SCHEMA OVERVIEW: {db_path} ---")
    
    for table_name_tuple in tables:
        table_name = table_name_tuple[0]
        
        # Skip internal SQLite system tables
        if table_name.startswith('sqlite_'):
            continue
            
        print(f"\nCLASS: {table_name}")
        print("-" * (len(table_name) + 7))

        # 2. Get information about each column (your "Fields")
        # PRAGMA table_info returns (id, name, type, notnull, default_value, pk)
        cursor.execute(f"PRAGMA table_info('{table_name}')")
        columns = cursor.fetchall()

        for col in columns:
            col_id = col[0]
            col_name = col[1]
            col_type = col[2]
            is_pk = " (Primary Key)" if col[5] == 1 else ""
            
            print(f"  [{col_id}] {col_name:<20} | Type: {col_type}{is_pk}")

    conn.close()

# Run it
explore_database_schema('../data/db_new.sqlite')

--- DATABASE SCHEMA OVERVIEW: ../data/db_new.sqlite ---

CLASS: Company
--------------
  [0] Id                   | Type: INTEGER (Primary Key)
  [1] Name                 | Type: TEXT

CLASS: Supplier
---------------
  [0] Id                   | Type: INTEGER (Primary Key)
  [1] Name                 | Type: TEXT
  [2] Ethics               | Type: TEXT
  [3] EsgScore             | Type: INTEGER

CLASS: Equivalence Class
------------------------
  [0] Id                   | Type: INTEGER (Primary Key)
  [1] Name                 | Type: TEXT

CLASS: BOM
----------
  [0] Id                   | Type: INTEGER (Primary Key)
  [1] ProducedProductId    | Type: INTEGER

CLASS: BOM_Component
--------------------
  [0] BOMId                | Type: INTEGER (Primary Key)
  [1] ConsumedProductId    | Type: INTEGER

CLASS: FinalProduct
-------------------
  [0] Id                   | Type: INTEGER (Primary Key)
  [1] CompanyId            | Type: INTEGER
  [2] Name                 | Type: TEXT

CLASS: 

In [ ]:

# Populates the new features of the database with educated guesses (randomly)
import sqlite3
import random

def populate_supplier_product_data(db_path):
    conn = sqlite3.connect(db_path)
    cursor = conn.cursor()

    # Sub-function 1: Random Price (Real)
    # Generates a price between 10.00 and 500.00
    def get_random_price():
        return round(random.uniform(10.0, 500.0), 2)

    # Sub-function 2: Random Quality (Int)
    # Matches your Quality D/C logic: 1 to 5 stars
    def get_random_quality():
        return random.randint(1, 5)

    # Sub-function 3: Random Place of Production (String)
    def get_random_place():
        places = ["Germany", "USA", "China", "Japan", "Mexico", "Vietnam"]
        return random.choice(places)

    # 1. Fetch all existing relationships
    cursor.execute("SELECT SupplierId, ProductId FROM Supplier_Product")
    pairs = cursor.fetchall()

    print(f"Updating {len(pairs)} records in Supplier_Product...")

    # 2. Iterate and Update
    for s_id, p_id in pairs:
        price = get_random_price()
        quality = get_random_quality()
        place = get_random_place()

        cursor.execute("""
            UPDATE Supplier_Product 
            SET Price = ?, Quality = ?, PlaceOfProduction = ?
            WHERE SupplierId = ? AND ProductId = ?
        """, (price, quality, place, s_id, p_id))

    conn.commit()
    conn.close()
    print("Database successfully populated with random test data!")

def check_random_entry(db_path):
    conn = sqlite3.connect(db_path)
    # This allows us to access columns by name like a dictionary
    conn.row_factory = sqlite3.Row 
    cursor = conn.cursor()

    # Get just one entry that has data in our new fields
    query = """
        SELECT * FROM Supplier_Product 
        WHERE Price IS NOT NULL 
        LIMIT 1
    """
    
    cursor.execute(query)
    row = cursor.fetchone()

    if row:
        print("--- SAMPLE ENTRY VERIFICATION ---")
        print(f"Supplier ID: {row['SupplierId']}")
        print(f"Product ID:  {row['ProductId']}")
        print(f"Price:       €{row['Price']:.2f}")
        print(f"Quality:     {row['Quality']}/5")
        print(f"Production:  {row['PlaceOfProduction']}")
    else:
        print("❌ No data found! The table is either empty or the update failed.")

    conn.close()


populate_supplier_product_data('../data/db_new.sqlite')
# Run it using the path we fixed earlier
check_random_entry('../data/db_new.sqlite')

In [ ]:
import sqlite3
import re
from collections import defaultdict

def group_by_functional_substance(db_path):
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()

    # 1. Fetch the data
    query = "SELECT p.SKU, sp.Price, sp.Quality FROM Product p JOIN Supplier_Product sp ON p.Id = sp.ProductId"
    cursor.execute(query)
    rows = cursor.fetchall()

    groups = defaultdict(list)

    # 2. Define functional keyword mapping (Internet/Domain Knowledge)
    # We look for these themes inside the messy SKU strings
    themes = {
        "Vitamin D": ["vitamin-d", "cholecalciferol", "vit-d"],
        "Magnesium": ["magnesium", "mg-"],
        "Protein": ["protein", "whey", "isolate", "bcaa"],
        "Electrolytes": ["electrolyte", "hydration", "salt", "sodium", "potassium"],
        "Multivitamin": ["multivitamin", "multi-", "once-daily"],
        "Calcium": ["calcium"],
        "Vitamin C": ["ascorbic-acid", "vitamin-c"],
        "B-Vitamins": ["vitamin-b", "b12", "niacin", "riboflavin", "thiamine"],
    }

    for row in rows:
        sku = row['SKU'].lower()
        found_theme = "Other/Specialty"
        
        # Check SKU against our functional themes
        for theme, keywords in themes.items():
            if any(k in sku for k in keywords):
                found_theme = theme
                break
        
        groups[found_theme].append({
            'sku': row['SKU'],
            'price': row['Price'] if row['Price'] else 0.0,
            'quality': row['Quality'] if row['Quality'] else 0
        })

    # 3. Print the Grouped Report
    print(f"{'FUNCTIONAL GROUP':<20} | {'SKU':<45} | {'PRICE':<8} | {'QUAL'}")
    print("="*90)

    for theme in sorted(groups.keys()):
        products = groups[theme]
        # Sort by price to show interchangeability options (cheapest first)
        sorted_products = sorted(products, key=lambda x: x['price'])
        
        for p in sorted_products:
            # Only print theme name for the first item in a block
            display_theme = theme if p == sorted_products[0] else ""
            print(f"{display_theme:<20} | {p['sku'][:45]:<45} | €{p['price']:>7.2f} | {p['quality']}/5")
        
        print("-" * 90)

    conn.close()

# Run the grouping
group_by_functional_substance('../data/db_new.sqlite')

In [ ]:
import sys
import os
import sqlite3
import re

# Adds the backend directory to your Python path
sys.path.append(os.path.abspath("../src/backend"))

# Now you can import your classes and types
from models import BOMEntry, BillOfMaterials
from component_from_supplier import ComponentFromSupplier

def fetch_random_bom_from_db(db_path: str) -> BillOfMaterials:
    sample_size = 1
    conn = sqlite3.connect(db_path)
    conn.row_factory = sqlite3.Row
    cursor = conn.cursor()

    # ORDER BY RANDOM() is the standard SQLite way to shuffle rows
    # LIMIT ensures we only get a specific number of items
    query = """
        SELECT component_id, equivalence_class, required_certs, forbidden_allergens 
        FROM BOM
        ORDER BY RANDOM() 
        LIMIT ?
    """
    
    cursor.execute(query, (sample_size,))
    rows = cursor.fetchall()

    bom: BillOfMaterials = []

    for row in rows:
        # Convert comma-separated strings from DB into tuples for the dataclass
        certs = tuple(c.strip() for c in row['required_certs'].split(',')) if row['required_certs'] else ()
        allergens = tuple(a.strip() for a in row['forbidden_allergens'].split(',')) if row['forbidden_allergens'] else ()

        entry = BOMEntry(
            component_id=str(row['component_id']),
            equivalence_class=str(row['equivalence_class']),
            required_certs=certs,
            forbidden_allergens=allergens
        )
        bom.append(entry)

    conn.close()
    return bom

bom = fetch_random_bom_from_db('../data/db_new.sqlite')
print(bom)

OperationalError: no such table: BillOfMaterials